# Notebook 4: Visualization and Analysis
## CS336 - Artificial Intelligence and Machine Learning (AIML)
### Assignment: Smart Energy Consumption Advisory Agent

**Purpose:** This notebook transforms model outputs into rich, informative visualisations  
to communicate findings to non-technical stakeh...

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (13, 5)})
print('Imports successful.')

## 1. Load Data

In [ ]:
# ─── Load the modelled dataset from Notebook 3 ───────────────────────────────
df = pd.read_csv('data_modelled.csv', parse_dates=['timestamp'], index_col='timestamp')
daily = pd.read_csv('data_daily.csv', parse_dates=['timestamp'], index_col='timestamp')

print(f'Loaded {len(df):,} rows. Date range: {df.index.min().date()} → {df.index.max().date()}')

## 2. Hourly Average Consumption Heatmap

A **day-of-week × hour heatmap** reveals when peak demand occurs, guiding load-shifting advice.

In [ ]:
# ─── Build pivot table: rows = day of week, columns = hour of day ────────────
df['dow']  = df.index.dayofweek   # 0 = Monday … 6 = Sunday
df['hour'] = df.index.hour        # 0 – 23

pivot = df.pivot_table(values='global_active_power',
                       index='dow', columns='hour', aggfunc='mean')

# Rename row index to human-readable day names
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
pivot.index = day_names

# Plot heatmap — warmer colours = higher average power consumption
plt.figure(figsize=(15, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Avg Power (kW)'})
plt.title('Average Power Consumption Heatmap (Day × Hour)', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

## 3. Average Diurnal Profile per Cluster

Plotting the **average hourly power curve** for each K-Means cluster reveals the  
characteristic usage shape that defines each behavioural group.

In [ ]:
# ─── Compute mean hourly profile for each cluster ────────────────────────────
cluster_profile = df.groupby(['cluster', 'hour'])['global_active_power'].mean().unstack(level=0)

# Descriptive labels for each cluster (based on centroid inspection from NB3)
cluster_labels = {
    0: 'Cluster 0 – Night Off-peak',
    1: 'Cluster 1 – Morning Ramp-up',
    2: 'Cluster 2 – Daytime Steady',
    3: 'Cluster 3 – Evening Peak'
}

plt.figure(figsize=(13, 6))
colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']  # Distinct colours per cluster

for col, color in zip(cluster_profile.columns, colors):
    plt.plot(cluster_profile.index,
             cluster_profile[col],
             label=cluster_labels.get(col, f'Cluster {col}'),
             linewidth=2.5, color=color)

plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Average Power (kW)', fontsize=12)
plt.title('Diurnal Power Profile per Cluster', fontsize=14)
plt.legend(fontsize=10)
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 4. Monthly Energy Consumption Trend

In [ ]:
# ─── Aggregate to monthly total and plot trend ───────────────────────────────
# Resample to monthly, sum all 15-min readings → proportional to kWh
monthly = df['global_active_power'].resample('ME').sum() * (15/60)  # Convert to kWh units

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(monthly.index, monthly.values,
              width=20, color='steelblue', edgecolor='white', alpha=0.85)

# Overlay a rolling trend line for context
ax.plot(monthly.index, monthly.rolling(3, min_periods=1).mean().values,
        color='darkred', linewidth=2, linestyle='--', label='3-Month Rolling Average')

ax.set_title('Monthly Energy Consumption (kWh)', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Energy (kWh)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sub-metering Breakdown

In [ ]:
# ─── Pie chart: average proportion of each sub-meter to total usage ──────────
sub_means = df[['sub_metering_1', 'sub_metering_2', 'sub_metering_3']].mean()

labels = ['Kitchen (Sub 1)', 'Laundry (Sub 2)', 'HVAC (Sub 3)']
colours = ['#FF6B6B', '#4ECDC4', '#45B7D1']

plt.figure(figsize=(7, 7))
wedges, texts, autotexts = plt.pie(
    sub_means, labels=labels, colors=colours,
    autopct='%1.1f%%', startangle=140,
    textprops={'fontsize': 12})
plt.title('Average Sub-metering Contribution to Total Usage', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Anomaly Calendar View

In [ ]:
# ─── Aggregate daily anomaly count and plot on a bar chart ───────────────────
daily_anomalies = df['is_anomaly'].resample('D').sum()  # Total anomalies per day

# Focus on first 3 months for a readable chart
da_q1 = daily_anomalies['2023-01':'2023-03']

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(da_q1.index, da_q1.values, color='tomato', edgecolor='white', width=0.8)
ax.set_title('Daily Anomaly Count (Jan–Mar 2023)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Anomalies per Day')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Cluster Distribution by Day of Week

In [ ]:
# ─── Stacked bar chart: cluster composition for each day of the week ─────────
dow_cluster = (df.groupby(['dow', 'cluster']).size()
                 .unstack(fill_value=0)
                 .apply(lambda x: x / x.sum(), axis=1))  # Normalise to proportions

dow_cluster.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

dow_cluster.plot(kind='bar', stacked=True, figsize=(11, 6),
                 colormap='tab10', edgecolor='white')

plt.title('Cluster Proportion by Day of Week', fontsize=13)
plt.xlabel('Day of Week')
plt.ylabel('Proportion of Intervals')
plt.legend(title='Cluster', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()